In [ ]:
def predict_score(hours):
    if hours <= 2.5:
        return 52.5
    else:
        return 82.5
    
print(predict_score(2))
print(predict_score(3.5))
print(predict_score(2.5))

52.5
82.5
52.5


In [5]:
def predict_score(hours):
    if hours <= 1.5:
        return 50
    else:
        return (55+80+85)/3
    
print(predict_score(1))
print(predict_score(3.5))
print(predict_score(2.5))
sse = (50-50)**2 + ((55+80+85)/3 - 55)**2 + ((55+80+85)/3 - 80)**2 + ((55+80+85)/3 - 85)**2
print(sse)

50
73.33333333333333
73.33333333333333
516.6666666666666


In [10]:
from sklearn.tree import DecisionTreeRegressor

X = [[1], [2], [3], [4]]
y = [50, 55, 80, 85]
tree = DecisionTreeRegressor(max_depth=1, random_state=42)
tree.fit(X, y)
print(tree.predict([[1], [3.5], [2.5]]))

[52.5 82.5 52.5]


- max_depth=1 allows one level of splitting: a root split with up to two leaves.
- random_state=42 makes any random tie-breaking reproducible.
- fit(X, y) learns the split and leaf predictions from the training data.

In [11]:
tree2 = DecisionTreeRegressor(max_depth=2, random_state=42)
tree2.fit(X, y)

print(tree2.predict(X))

[50. 55. 80. 85.]


In [13]:
import numpy as np

rng = np.random.default_rng(6)
X = rng.uniform(-3, 3, size=(80, 1))
noise = rng.normal(0, 3, size=80)
y = 3 * X[:, 0]**2 + noise

from sklearn.model_selection import train_test_split

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_dev.shape, X_test.shape)

(64, 1) (16, 1)


- rng: makes our random data reproducible.
- X: generates 80 observations, each with one feature.
- noise: generates random noise with standard deviation 3.
- y: combines the underlying relationship with noise.

In [14]:
from sklearn.model_selection import cross_val_score

shallow_tree = DecisionTreeRegressor(max_depth=1, random_state=42)

scores = cross_val_score(
    shallow_tree, X_dev, y_dev,
    cv=4, scoring="neg_mean_squared_error"
)

fold_mse = -scores
print(fold_mse)
print(fold_mse.mean())

[ 63.08010509 110.39886351  62.90569831  34.56425693]
67.73723095971752


- max_depth=1: the tree can make only one split, giving at most two leaf predictions.
- cv=4: each round uses 48 observations for training and 16 for validation.
- fold_mse = -scores: converts sklearn’s negative MSE scores back to ordinary MSE.

In [15]:
medium_tree = DecisionTreeRegressor(max_depth=3, random_state=42)

scores = cross_val_score(
    medium_tree, X_dev, y_dev,
    cv=4, scoring="neg_mean_squared_error"
)

fold_mse = -scores
print(fold_mse)
print(fold_mse.mean())

[14.57835203 33.25223972 28.20623493 13.13440811]
22.292808697118424


In [16]:
deep_tree = DecisionTreeRegressor(max_depth=None, random_state=42)

scores = cross_val_score(
    deep_tree, X_dev, y_dev,
    cv=4, scoring="neg_mean_squared_error"
)

fold_mse = -scores
print(fold_mse)
print(fold_mse.mean())

[14.70083598 33.86852467 32.47719679 17.79153161]
24.709522263973106


In [18]:
from sklearn.metrics import mean_squared_error

final_tree = DecisionTreeRegressor(max_depth=3)
final_tree.fit(X_dev, y_dev)

y_test_pred = final_tree.predict(X_test)
test_mse = mean_squared_error(y_test, y_test_pred)

print(test_mse)

14.859388178954454


- Why did you choose a maximum depth of 3 instead of an unrestricted tree?
- I chose a maximum depth of 3 because it had the lowest mean cross-validation MSE among the models I tested. Removing the depth limit increased the CV MSE from 22.29 to 24.71, suggesting that the additional complexity did not improve generalization.